# HQDeepDTAF — Full Training on PDBbind v2016

**Paper:** Jeong et al., *Hybrid quantum neural networks for efficient protein-ligand binding affinity prediction*, EPJ Quantum Technology (2025) 12:120

## Before running this notebook, upload to your Google Drive:

```
MyDrive/HybridQNN/
├── metrics.py
├── dataset.py
├── model.py
├── preprocess.py
├── train.py
└── pdbbind2016/
    ├── v2016-core/
    ├── v2016-other-PL/
    └── index/
```

**Session timeout?** Re-run the notebook — training resumes from the last completed epoch automatically.

In [ ]:
# Cell 1 — Install all dependencies
# dssp is the mkdssp binary used by BioPython for secondary structure prediction
!apt-get install -y dssp 2>/dev/null | tail -1
!pip install torch pennylane biopython openbabel-wheel numpy tqdm --quiet
print('All packages installed.')

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Configure paths
# Edit DRIVE_ROOT if your folder is named differently.
# Everything else is derived from it automatically.

DRIVE_ROOT    = '/content/drive/MyDrive/HybridQNN'
PDBBIND_DIR   = DRIVE_ROOT + '/pdbbind2016'
PROCESSED_DIR = DRIVE_ROOT + '/data/processed'
SPLITS_DIR    = DRIVE_ROOT + '/data/splits'
CKPT_DIR      = DRIVE_ROOT + '/checkpoints'

import sys, os
sys.path.insert(0, DRIVE_ROOT)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print('Paths configured:')
print('  Project root :', DRIVE_ROOT)
print('  PDBbind data :', PDBBIND_DIR)
print('  Processed    :', PROCESSED_DIR)
print('  Splits       :', SPLITS_DIR)
print('  Checkpoints  :', CKPT_DIR)

In [ ]:
# Cell 4 — Verify source files are present in Drive
import os

required = ['metrics.py', 'dataset.py', 'model.py', 'preprocess.py', 'train.py']
missing = [f for f in required if not os.path.exists(os.path.join(DRIVE_ROOT, f))]

if missing:
    print('ERROR — missing files in', DRIVE_ROOT)
    for m in missing:
        print('  MISSING:', m)
    print()
    print('Upload the missing .py files from your local HybridQNN/ folder to Google Drive.')
    raise FileNotFoundError('Upload source files first.')
else:
    for f in required:
        print('  OK  ', f)
    print('All source files found.')

In [ ]:
# Cell 5 — Verify PDBbind v2016 directory structure
import os

CORE_SUBDIR    = 'v2016-core'
GENERAL_SUBDIR = 'v2016-other-PL'

checks = {
    'Core set dir'   : os.path.join(PDBBIND_DIR, CORE_SUBDIR),
    'General set dir': os.path.join(PDBBIND_DIR, GENERAL_SUBDIR),
    'Core index'     : os.path.join(PDBBIND_DIR, 'index/INDEX_core_data.2016'),
    'General index'  : os.path.join(PDBBIND_DIR, 'index/INDEX_general_PL_data.2016'),
}

all_ok = True
for name, path in checks.items():
    ok = os.path.exists(path)
    print('  %s  %-20s %s' % ('OK ' if ok else 'ERR', name, path))
    if not ok:
        all_ok = False

core_dir = os.path.join(PDBBIND_DIR, CORE_SUBDIR)
general_dir = os.path.join(PDBBIND_DIR, GENERAL_SUBDIR)

if os.path.isdir(core_dir):
    n_core = len([d for d in os.listdir(core_dir) if os.path.isdir(os.path.join(core_dir, d))])
    print('\n  Core complexes    :', n_core, '(expected ~290)')

if os.path.isdir(general_dir):
    n_gen = len([d for d in os.listdir(general_dir) if os.path.isdir(os.path.join(general_dir, d))])
    print('  General complexes :', n_gen, '(expected ~3700)')

if not all_ok:
    raise FileNotFoundError('Fix missing PDBbind paths before continuing.')

## Preprocessing

Converts raw PDB/SDF files to `.npy` feature arrays.

- **Estimated time:** 2–4 hours for ~4,000 complexes
- **Output:** saved directly to Google Drive (safe to reconnect if session drops)
- **If preprocessing was already run:** skip to the Training section

Per complex, writes four files:
```
data/processed/<pdb_id>_seq.npy   float32 (1000, 40)
data/processed/<pdb_id>_pkt.npy   float32 (63, 40)
data/processed/<pdb_id>_smi.npy   int64   (150,)
data/processed/<pdb_id>_aff.npy   float32 scalar
```

In [ ]:
# Cell 6 — Check whether preprocessing has already been done
import os, glob

train_split = os.path.join(SPLITS_DIR, 'train.txt')
test_split  = os.path.join(SPLITS_DIR, 'test.txt')
n_processed = len(glob.glob(os.path.join(PROCESSED_DIR, '*_seq.npy')))

print('Processed complexes found :', n_processed)
print('train.txt exists          :', os.path.exists(train_split))
print('test.txt  exists          :', os.path.exists(test_split))

if n_processed > 100 and os.path.exists(train_split) and os.path.exists(test_split):
    print()
    print('Preprocessing looks complete. Skip Cell 7 and go to Training.')
else:
    print()
    print('Preprocessing needed — run Cell 7.')

In [ ]:
# Cell 7 — Run preprocessing
# Skip this cell if Cell 6 shows preprocessing is already complete.
import subprocess, sys, os

# Change to DRIVE_ROOT so relative paths (data/splits/) resolve correctly
os.chdir(DRIVE_ROOT)

cmd = [
    sys.executable,
    os.path.join(DRIVE_ROOT, 'preprocess.py'),
    '--pdbbind_dir', PDBBIND_DIR,
    '--output_dir', PROCESSED_DIR,
    '--core_subdir', 'v2016-core',
    '--general_subdir', 'v2016-other-PL',
]

print('Starting preprocessing...')
print('Estimated time: 2-4 hours. Outputs saved to Drive continuously.')
print('Command:', ' '.join(cmd))
print()

result = subprocess.run(cmd, text=True)
print('\nPreprocessing finished. Return code:', result.returncode)

# Move split files if preprocess.py wrote them to cwd/data/splits/
import shutil
for split in ('train.txt', 'test.txt'):
    src = os.path.join(DRIVE_ROOT, 'data', 'splits', split)
    dst = os.path.join(SPLITS_DIR, split)
    if os.path.exists(src) and src != dst:
        shutil.move(src, dst)
        print('Moved', split, 'to', SPLITS_DIR)

print('Train split:', os.path.join(SPLITS_DIR, 'train.txt'))
print('Test  split:', os.path.join(SPLITS_DIR, 'test.txt'))

In [ ]:
# Cell 8 — Verify preprocessing output
import os, glob

n_seq = len(glob.glob(os.path.join(PROCESSED_DIR, '*_seq.npy')))
n_pkt = len(glob.glob(os.path.join(PROCESSED_DIR, '*_pkt.npy')))
n_smi = len(glob.glob(os.path.join(PROCESSED_DIR, '*_smi.npy')))
n_aff = len(glob.glob(os.path.join(PROCESSED_DIR, '*_aff.npy')))

print('Processed feature files:')
print('  _seq.npy :', n_seq)
print('  _pkt.npy :', n_pkt)
print('  _smi.npy :', n_smi)
print('  _aff.npy :', n_aff)

for split_name, split_file in (('train', os.path.join(SPLITS_DIR, 'train.txt')),
                                ('test',  os.path.join(SPLITS_DIR, 'test.txt'))):
    if os.path.exists(split_file):
        with open(split_file) as f:
            ids = [l.strip() for l in f if l.strip()]
        print('%s split: %d IDs' % (split_name, len(ids)))
    else:
        print('%s split: NOT FOUND' % split_name)

# Spot-check one record
import numpy as np
train_split = os.path.join(SPLITS_DIR, 'train.txt')
if os.path.exists(train_split):
    with open(train_split) as f:
        first_id = f.readline().strip()
    if first_id:
        seq = np.load(os.path.join(PROCESSED_DIR, first_id + '_seq.npy'))
        pkt = np.load(os.path.join(PROCESSED_DIR, first_id + '_pkt.npy'))
        smi = np.load(os.path.join(PROCESSED_DIR, first_id + '_smi.npy'))
        aff = np.load(os.path.join(PROCESSED_DIR, first_id + '_aff.npy'))
        print('\nSpot-check [%s]:' % first_id)
        print('  seq shape:', seq.shape, 'dtype:', seq.dtype)
        print('  pkt shape:', pkt.shape, 'dtype:', pkt.dtype)
        print('  smi shape:', smi.shape, 'dtype:', smi.dtype)
        print('  affinity :', float(aff))

## Training

- 5 independent runs × 20 epochs each (paper protocol)
- Checkpoints saved to Drive after **every epoch** — resume safely after timeout
- Best model (lowest train MSE across runs) saved to `checkpoints/best_model.pt`
- **To resume:** simply re-run this cell — it reads `progress.json` and picks up where it left off

Paper target results (HQDeepDTAF-NN-Angle, 9 qubits):

| MAE | RMSE | R | SD | CI |
|-----|------|---|----|----|  
| 1.082 | 1.368 | 0.783 | 1.355 | 0.792 |

In [ ]:
# Cell 9 — Training with epoch-level checkpointing
# Re-run this cell after any session timeout to resume automatically.

import torch
import torch.nn as nn
import json
import os
from torch.utils.data import DataLoader
from tqdm import tqdm

# Fresh module import from Drive
import importlib, sys
for mod in ('metrics', 'dataset', 'model'):
    if mod in sys.modules:
        del sys.modules[mod]

from model import DeepDTAF, test
from dataset import PDBbindDataset

# ── Hyperparameters (match paper) ──────────────────────────────────────────
EPOCHS       = 20
RUNS         = 5
BATCH_SIZE   = 16
LR           = 0.005
WEIGHT_DECAY = 0.01
NUM_WORKERS  = 2       # Colab has 2 CPU cores by default

PROGRESS_FILE = os.path.join(CKPT_DIR, 'progress.json')
device = torch.device('cpu')

# ── Load datasets ──────────────────────────────────────────────────────────
train_ds = PDBbindDataset(PROCESSED_DIR, os.path.join(SPLITS_DIR, 'train.txt'))
test_ds  = PDBbindDataset(PROCESSED_DIR, os.path.join(SPLITS_DIR, 'test.txt'))
print('Train: %d complexes  |  Test: %d complexes' % (len(train_ds), len(test_ds)))

# drop_last=True prevents batch-size-1 which breaks Squeeze()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, drop_last=True)
loss_fn = nn.MSELoss()

# ── Load or initialise progress ────────────────────────────────────────────
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        progress = json.load(f)
    print('Resuming: run %d/%d, epoch %d/%d' % (
        progress['current_run'] + 1, RUNS,
        progress['current_epoch'] + 1, EPOCHS))
else:
    progress = {
        'current_run'   : 0,
        'current_epoch' : 0,
        'all_results'   : [],
        'best_run_loss' : 1e18,
    }
    print('Starting fresh training.')

def save_progress():
    with open(PROGRESS_FILE, 'w') as pf:
        json.dump(progress, pf, indent=2)

# ── Training loop ──────────────────────────────────────────────────────────
for run_id in range(progress['current_run'], RUNS):
    model     = DeepDTAF().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    ckpt_path       = os.path.join(CKPT_DIR, 'run%d_latest.pt' % run_id)
    start_epoch     = 1
    best_train_loss = 1e18
    best_eval       = None

    # Resume mid-run if a checkpoint exists for this run
    mid_run = (run_id == progress['current_run'] and progress['current_epoch'] > 0)
    if os.path.exists(ckpt_path) and mid_run:
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        start_epoch     = ckpt['epoch'] + 1
        best_train_loss = ckpt['best_train_loss']
        best_eval       = ckpt.get('best_eval')
        print('  Resumed run %d from epoch %d' % (run_id + 1, start_epoch))

    print('\n--- Run %d/%d  (epochs %d-%d) ---' % (run_id + 1, RUNS, start_epoch, EPOCHS))

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        epoch_loss, n_samples = 0.0, 0

        for seq, pkt, smi, label in tqdm(train_loader,
                                         desc='  Epoch %2d' % epoch,
                                         leave=False):
            seq   = seq.to(device)
            pkt   = pkt.to(device)
            smi   = smi.to(device)
            label = label.to(device)
            pred  = model(seq, pkt, smi)
            loss  = loss_fn(pred.view(-1), label.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(label)
            n_samples  += len(label)

        train_loss = epoch_loss / n_samples
        print('  Epoch %2d/%d  train_MSE=%.4f' % (epoch, EPOCHS, train_loss), end='')

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            eval_result     = test(model, test_loader, loss_fn, device, show=False)
            best_eval       = eval_result
            print('  <- best | MAE=%.3f  RMSE=%.3f  R=%.3f' % (
                eval_result['MAE'], eval_result['RMSE'], eval_result['CORR']), end='')
        print()

        # Save epoch checkpoint to Drive (overwrites previous for this run)
        torch.save({
            'epoch'          : epoch,
            'model'          : model.state_dict(),
            'optimizer'      : optimizer.state_dict(),
            'best_train_loss': best_train_loss,
            'best_eval'      : best_eval,
        }, ckpt_path)
        progress['current_epoch'] = epoch
        save_progress()

    # Run complete — record results
    progress['all_results'].append(best_eval)
    if best_train_loss < progress['best_run_loss']:
        progress['best_run_loss'] = best_train_loss
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, 'best_model.pt'))
        print('  -> Best model saved (run %d)' % (run_id + 1))

    progress['current_run']    = run_id + 1
    progress['current_epoch']  = 0
    save_progress()

# ── Final results ──────────────────────────────────────────────────────────
all_results = [r for r in progress['all_results'] if r is not None]
n_runs = len(all_results)

print('\n========== Final Results (averaged over %d runs) ==========' % n_runs)
for key in ('MAE', 'RMSE', 'CORR', 'SD', 'c_index'):
    vals = [r[key] for r in all_results]
    if vals:
        print('  %-8s: %.4f' % (key, sum(vals) / len(vals)))

print('\nPaper target (HQDeepDTAF-NN-Angle, 9 qubits):')
print('  MAE=1.082  RMSE=1.368  R=0.783  SD=1.355  CI=0.792')
print('\nBest model saved to:', os.path.join(CKPT_DIR, 'best_model.pt'))

In [ ]:
# Cell 10 — Per-run breakdown
import json, os

PROGRESS_FILE = os.path.join(CKPT_DIR, 'progress.json')

if not os.path.exists(PROGRESS_FILE):
    print('No progress file found — run training first.')
else:
    with open(PROGRESS_FILE) as f:
        progress = json.load(f)

    results = [r for r in progress['all_results'] if r is not None]

    print('%-6s  %-7s  %-7s  %-7s  %-7s  %-7s' % (
        'Run', 'MAE', 'RMSE', 'R', 'SD', 'CI'))
    print('-' * 52)
    for i, r in enumerate(results):
        print('%-6d  %-7.4f  %-7.4f  %-7.4f  %-7.4f  %-7.4f' % (
            i + 1,
            r['MAE'], r['RMSE'], r['CORR'], r['SD'], r['c_index']))

    if results:
        print('-' * 52)
        print('%-6s  %-7.4f  %-7.4f  %-7.4f  %-7.4f  %-7.4f' % (
            'Mean',
            sum(r['MAE']     for r in results) / len(results),
            sum(r['RMSE']    for r in results) / len(results),
            sum(r['CORR']    for r in results) / len(results),
            sum(r['SD']      for r in results) / len(results),
            sum(r['c_index'] for r in results) / len(results),
        ))